# Notebook 01 — Generate Synthetic Transactions

## Fraud Graph Analytics  
### Geração de Dados Sintéticos para Prevenção a Fraudes Transacionais

Este notebook tem como objetivo criar uma base sintética de transações financeiras para sustentar o MVP de prevenção a fraudes com **Knowledge Graph**, **regras explicáveis** e **análise de redes**.

A geração dos dados será orientada pela etapa anterior de **Domain Understanding**, contemplando entidades como:

- clientes;
- contas;
- transações;
- dispositivos;
- IPs;
- beneficiários;
- cartões;
- cenários de fraude;
- labels analíticos.

Os dados sintéticos serão utilizados nos próximos notebooks para análise exploratória, motor de regras antifraude, modelagem em grafo e score de risco explicável.

## 1. Objetivo da Célula

### Objetivo

Configurar o ambiente inicial do notebook, importar bibliotecas, definir caminhos do projeto e preparar as configurações globais de geração sintética.

### Ações realizadas

- Importação das bibliotecas principais.
- Configuração de seed para reprodutibilidade.
- Definição dos diretórios de entrada e saída.
- Definição dos parâmetros principais de volume dos dados.

### Justificativa técnica

Como este projeto será usado em portfólio, a geração dos dados precisa ser reprodutível, organizada e facilmente ajustável. O uso de uma seed fixa permite que os mesmos dados sejam recriados em diferentes execuções, garantindo consistência entre notebooks.

### Resultados esperados

Ambiente preparado para gerar as entidades sintéticas do domínio transacional antifraude.

In [1]:
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from faker import Faker

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

SEED = 42
rng = np.random.default_rng(SEED)
fake = Faker("pt_BR")
Faker.seed(SEED)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "synthetic"
DOCS_DIR = PROJECT_ROOT / "docs"

SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:   {PROJECT_ROOT}")
print(f"Synthetic dir:  {SYNTHETIC_DIR}")
print(f"Docs dir:       {DOCS_DIR}")

Project root:   d:\_DS-Projects\Data-Science\fraud-graph-analytics
Synthetic dir:  d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\synthetic
Docs dir:       d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs


## 2. Parâmetros de Geração

Nesta etapa definimos os volumes iniciais da base sintética.

A proposta é criar um dataset suficientemente robusto para permitir análises interessantes, mas ainda leve o bastante para execução local com Pandas, DuckDB, NetworkX e Neo4j em ambiente de portfólio.

Os volumes podem ser ajustados posteriormente conforme a necessidade do projeto.

In [2]:
CONFIG = {
    "n_clientes": 5_000,
    "n_contas": 6_000,
    "n_dispositivos": 4_500,
    "n_ips": 3_000,
    "n_beneficiarios": 3_500,
    "n_cartoes": 4_000,
    "n_transacoes": 80_000,
    "data_inicio": datetime(2025, 1, 1),
    "data_fim": datetime(2025, 12, 31, 23, 59, 59),
}

CONFIG

{'n_clientes': 5000,
 'n_contas': 6000,
 'n_dispositivos': 4500,
 'n_ips': 3000,
 'n_beneficiarios': 3500,
 'n_cartoes': 4000,
 'n_transacoes': 80000,
 'data_inicio': datetime.datetime(2025, 1, 1, 0, 0),
 'data_fim': datetime.datetime(2025, 12, 31, 23, 59, 59)}

## 3. Geração da Base de Clientes

A base de clientes representa pessoas físicas simuladas que possuem uma ou mais contas transacionais.

Nesta primeira versão, os clientes terão atributos simples, mas suficientes para análises cadastrais e geração de comportamento transacional:

- `cliente_id`;
- `idade`;
- `uf`;
- `segmento`;
- `data_cadastro`;
- `score_cadastral`.

O `score_cadastral` será um indicador sintético entre 0 e 1000, simulando uma visão cadastral/risk-based simplificada.

In [3]:
def random_dates(start: datetime, end: datetime, size: int) -> pd.Series:
    start_ts = int(start.timestamp())
    end_ts = int(end.timestamp())
    random_ts = rng.integers(start_ts, end_ts, size=size)
    return pd.to_datetime(random_ts, unit="s")


ufs = ["SP", "RJ", "MG", "PR", "SC", "RS", "BA", "PE", "GO", "DF", "CE", "ES"]
segmentos = ["varejo", "alta_renda", "universitario", "mei", "aposentado"]

clientes = pd.DataFrame(
    {
        "cliente_id": [f"CLI_{i:06d}" for i in range(1, CONFIG["n_clientes"] + 1)],
        "idade": rng.integers(18, 78, size=CONFIG["n_clientes"]),
        "uf": rng.choice(ufs, size=CONFIG["n_clientes"], p=[0.36, 0.12, 0.11, 0.08, 0.06, 0.06, 0.05, 0.04, 0.04, 0.03, 0.03, 0.02]),
        "segmento": rng.choice(segmentos, size=CONFIG["n_clientes"], p=[0.55, 0.12, 0.13, 0.14, 0.06]),
        "data_cadastro": random_dates(datetime(2020, 1, 1), datetime(2025, 12, 1), CONFIG["n_clientes"]).date,
        "score_cadastral": np.clip(rng.normal(650, 130, CONFIG["n_clientes"]).round(0), 150, 950).astype(int),
    }
)

clientes.head()

,cliente_id,idade,uf,segmento,data_cadastro,score_cadastral
0,CLI_000001,23,SP,varejo,2025-05-23,836
1,CLI_000002,64,SC,universitario,2024-05-09,879
2,CLI_000003,57,CE,varejo,2022-06-15,693
3,CLI_000004,44,BA,varejo,2020-06-22,490
4,CLI_000005,43,SP,varejo,2020-11-26,880


## 4. Geração de Contas

A tabela de contas representa contas transacionais vinculadas aos clientes.

Uma mesma pessoa pode ter mais de uma conta, o que permite simular comportamentos mais próximos de ambientes financeiros reais.

Atributos principais:

- `conta_id`;
- `cliente_id`;
- `tipo_conta`;
- `data_abertura`;
- `status_conta`;
- `limite_transacional_diario`.

In [4]:
contas = pd.DataFrame(
    {
        "conta_id": [f"CTA_{i:06d}" for i in range(1, CONFIG["n_contas"] + 1)],
        "cliente_id": rng.choice(clientes["cliente_id"], size=CONFIG["n_contas"], replace=True),
        "tipo_conta": rng.choice(["corrente", "pagamento", "digital"], size=CONFIG["n_contas"], p=[0.45, 0.25, 0.30]),
        "data_abertura": random_dates(datetime(2021, 1, 1), datetime(2025, 12, 20), CONFIG["n_contas"]).date,
        "status_conta": rng.choice(["ativa", "bloqueada", "encerrada"], size=CONFIG["n_contas"], p=[0.94, 0.04, 0.02]),
        "limite_transacional_diario": rng.choice([1_000, 2_000, 5_000, 10_000, 20_000, 50_000], size=CONFIG["n_contas"], p=[0.12, 0.20, 0.28, 0.23, 0.12, 0.05]),
    }
)

contas.head()

,conta_id,cliente_id,tipo_conta,data_abertura,status_conta,limite_transacional_diario
0,CTA_000001,CLI_004979,digital,2021-11-05,ativa,10000
1,CTA_000002,CLI_004278,corrente,2021-12-31,ativa,20000
2,CTA_000003,CLI_002410,corrente,2025-04-07,ativa,2000
3,CTA_000004,CLI_001673,digital,2025-09-02,ativa,20000
4,CTA_000005,CLI_002338,digital,2023-08-13,ativa,10000


## 5. Geração de Dispositivos, IPs, Beneficiários e Cartões

Nesta etapa serão criadas entidades auxiliares que serão conectadas às transações.

Essas entidades são fundamentais para o futuro Knowledge Graph, pois permitem investigar conexões indiretas entre contas, transações e comportamentos suspeitos.

Exemplos:

- várias contas usando o mesmo dispositivo;
- muitos clientes acessando por IPs recorrentes;
- beneficiários recebendo valores de múltiplas origens;
- cartões associados a diferentes padrões de uso.

In [5]:
dispositivos = pd.DataFrame(
    {
        "device_id": [f"DEV_{i:06d}" for i in range(1, CONFIG["n_dispositivos"] + 1)],
        "tipo_device": rng.choice(["mobile", "desktop", "tablet"], size=CONFIG["n_dispositivos"], p=[0.78, 0.18, 0.04]),
        "sistema_operacional": rng.choice(["Android", "iOS", "Windows", "Linux", "macOS"], size=CONFIG["n_dispositivos"], p=[0.48, 0.28, 0.17, 0.03, 0.04]),
        "fingerprint_risco": rng.choice(["baixo", "medio", "alto"], size=CONFIG["n_dispositivos"], p=[0.82, 0.14, 0.04]),
    }
)

ips = pd.DataFrame(
    {
        "ip_id": [f"IP_{i:06d}" for i in range(1, CONFIG["n_ips"] + 1)],
        "uf_origem": rng.choice(ufs, size=CONFIG["n_ips"]),
        "tipo_rede": rng.choice(["residencial", "movel", "corporativa", "vpn_proxy"], size=CONFIG["n_ips"], p=[0.48, 0.34, 0.12, 0.06]),
        "risco_rede": rng.choice(["baixo", "medio", "alto"], size=CONFIG["n_ips"], p=[0.80, 0.15, 0.05]),
    }
)

beneficiarios = pd.DataFrame(
    {
        "beneficiario_id": [f"BEN_{i:06d}" for i in range(1, CONFIG["n_beneficiarios"] + 1)],
        "tipo_beneficiario": rng.choice(["pessoa_fisica", "pessoa_juridica", "conta_interna"], size=CONFIG["n_beneficiarios"], p=[0.68, 0.22, 0.10]),
        "banco_destino": rng.choice(["banco_a", "banco_b", "banco_c", "banco_d", "mesma_instituicao"], size=CONFIG["n_beneficiarios"], p=[0.24, 0.22, 0.18, 0.16, 0.20]),
        "uf_destino": rng.choice(ufs, size=CONFIG["n_beneficiarios"]),
    }
)

cartoes = pd.DataFrame(
    {
        "cartao_id": [f"CAR_{i:06d}" for i in range(1, CONFIG["n_cartoes"] + 1)],
        "conta_id": rng.choice(contas["conta_id"], size=CONFIG["n_cartoes"], replace=True),
        "tipo_cartao": rng.choice(["debito", "credito", "virtual"], size=CONFIG["n_cartoes"], p=[0.43, 0.37, 0.20]),
        "status_cartao": rng.choice(["ativo", "bloqueado", "cancelado"], size=CONFIG["n_cartoes"], p=[0.91, 0.06, 0.03]),
        "data_emissao": random_dates(datetime(2021, 1, 1), datetime(2025, 12, 1), CONFIG["n_cartoes"]).date,
    }
)

display(dispositivos.head())
display(ips.head())
display(beneficiarios.head())
display(cartoes.head())

,device_id,tipo_device,sistema_operacional,fingerprint_risco
0,DEV_000001,mobile,iOS,baixo
1,DEV_000002,mobile,macOS,baixo
2,DEV_000003,mobile,Android,medio
3,DEV_000004,mobile,iOS,baixo
4,DEV_000005,mobile,iOS,baixo


,ip_id,uf_origem,tipo_rede,risco_rede
0,IP_000001,MG,movel,baixo
1,IP_000002,SC,movel,alto
2,IP_000003,PE,movel,baixo
3,IP_000004,DF,movel,baixo
4,IP_000005,PR,residencial,baixo


,beneficiario_id,tipo_beneficiario,banco_destino,uf_destino
0,BEN_000001,pessoa_fisica,banco_a,RJ
1,BEN_000002,pessoa_fisica,banco_a,BA
2,BEN_000003,conta_interna,banco_a,BA
3,BEN_000004,conta_interna,banco_d,PE
4,BEN_000005,conta_interna,banco_c,BA


,cartao_id,conta_id,tipo_cartao,status_cartao,data_emissao
0,CAR_000001,CTA_003914,debito,ativo,2025-07-05
1,CAR_000002,CTA_002604,debito,ativo,2025-05-13
2,CAR_000003,CTA_002446,debito,ativo,2021-01-22
3,CAR_000004,CTA_004727,credito,ativo,2025-10-15
4,CAR_000005,CTA_005590,virtual,ativo,2022-08-05


## 6. Geração da Base Transacional Normal

Agora será criada a base inicial de transações com comportamento majoritariamente normal.

Nesta versão, cada transação terá:

- conta de origem;
- beneficiário;
- valor;
- data e hora;
- tipo de transação;
- canal;
- dispositivo;
- IP;
- cartão opcional;
- status;
- label inicial de fraude igual a zero.

Depois desta etapa, padrões suspeitos serão injetados artificialmente para criar cenários controlados de fraude.

In [6]:
def generate_transaction_values(size: int) -> np.ndarray:
    values = rng.lognormal(mean=5.6, sigma=1.05, size=size)
    values = np.clip(values, 5, 25_000)
    return np.round(values, 2)


transacoes = pd.DataFrame(
    {
        "transacao_id": [f"TX_{i:08d}" for i in range(1, CONFIG["n_transacoes"] + 1)],
        "conta_origem_id": rng.choice(contas["conta_id"], size=CONFIG["n_transacoes"], replace=True),
        "beneficiario_id": rng.choice(beneficiarios["beneficiario_id"], size=CONFIG["n_transacoes"], replace=True),
        "valor": generate_transaction_values(CONFIG["n_transacoes"]),
        "data_hora": random_dates(CONFIG["data_inicio"], CONFIG["data_fim"], CONFIG["n_transacoes"]),
        "tipo_transacao": rng.choice(["pix", "ted", "boleto", "cartao", "transferencia_interna"], size=CONFIG["n_transacoes"], p=[0.55, 0.08, 0.14, 0.16, 0.07]),
        "canal": rng.choice(["app", "internet_banking", "atm", "agencia", "api"], size=CONFIG["n_transacoes"], p=[0.70, 0.14, 0.07, 0.04, 0.05]),
        "device_id": rng.choice(dispositivos["device_id"], size=CONFIG["n_transacoes"], replace=True),
        "ip_id": rng.choice(ips["ip_id"], size=CONFIG["n_transacoes"], replace=True),
        "status_transacao": rng.choice(["aprovada", "negada", "em_analise"], size=CONFIG["n_transacoes"], p=[0.94, 0.04, 0.02]),
        "is_fraud": 0,
        "fraud_scenario": "normal",
    }
)

cartoes_por_conta = cartoes.groupby("conta_id")["cartao_id"].apply(list).to_dict()

def assign_card(row):
    if row["tipo_transacao"] != "cartao":
        return None
    cards = cartoes_por_conta.get(row["conta_origem_id"], [])
    if not cards:
        return None
    return rng.choice(cards)

transacoes["cartao_id"] = transacoes.apply(assign_card, axis=1)

transacoes.head()

,transacao_id,conta_origem_id,beneficiario_id,valor,data_hora,tipo_transacao,canal,device_id,ip_id,status_transacao,is_fraud,fraud_scenario,cartao_id
0,TX_00000001,CTA_005927,BEN_000460,99.20,2025-10-22 22:17:42,pix,atm,DEV_003637,IP_000415,aprovada,0,normal,None
1,TX_00000002,CTA_001877,BEN_000751,135.32,2025-05-29 02:32:48,pix,app,DEV_002443,IP_001522,aprovada,0,normal,None
2,TX_00000003,CTA_000955,BEN_001031,447.55,2025-09-29 07:22:15,pix,app,DEV_001578,IP_001468,aprovada,0,normal,None
3,TX_00000004,CTA_004607,BEN_000351,534.86,2025-10-04 21:18:15,cartao,internet_banking,DEV_002654,IP_000789,aprovada,0,normal,None
4,TX_00000005,CTA_003730,BEN_001331,842.44,2025-03-31 08:34:54,pix,api,DEV_001129,IP_000180,aprovada,0,normal,None


## 7. Injeção de Cenários Suspeitos

Nesta etapa serão injetados padrões controlados de fraude.

A ideia não é criar dados reais, mas sim criar **sinais analíticos coerentes** com o domínio de prevenção a fraudes.

Cenários simulados:

1. `shared_device_ring` — várias contas usando o mesmo dispositivo;
2. `beneficiary_concentrator` — muitos envios para o mesmo beneficiário;
3. `new_account_high_value` — contas novas realizando transações de alto valor;
4. `burst_transactions` — múltiplas transações em curto período;
5. `bridge_account` — contas atuando como intermediárias;
6. `coordinated_network` — grupo coordenado com dispositivo, IP e beneficiário compartilhados.

In [7]:
transacoes_fraud = transacoes.copy()

# Seletores auxiliares
active_accounts = contas.loc[contas["status_conta"] == "ativa", "conta_id"].to_numpy()
recent_accounts = contas.loc[pd.to_datetime(contas["data_abertura"]) >= pd.Timestamp("2025-10-01"), "conta_id"].to_numpy()

high_risk_devices = dispositivos.loc[dispositivos["fingerprint_risco"] == "alto", "device_id"].to_numpy()
high_risk_ips = ips.loc[ips["risco_rede"] == "alto", "ip_id"].to_numpy()

if len(high_risk_devices) == 0:
    high_risk_devices = dispositivos["device_id"].sample(20, random_state=SEED).to_numpy()

if len(high_risk_ips) == 0:
    high_risk_ips = ips["ip_id"].sample(20, random_state=SEED).to_numpy()

def mark_scenario(indexes, scenario_name):
    transacoes_fraud.loc[indexes, "is_fraud"] = 1
    transacoes_fraud.loc[indexes, "fraud_scenario"] = scenario_name
    transacoes_fraud.loc[indexes, "status_transacao"] = rng.choice(
        ["aprovada", "em_analise", "negada"],
        size=len(indexes),
        p=[0.60, 0.30, 0.10],
    )

# 1. Shared device ring
shared_device_idx = rng.choice(transacoes_fraud.index, size=1_600, replace=False)
shared_devices = rng.choice(high_risk_devices, size=25, replace=True)
transacoes_fraud.loc[shared_device_idx, "device_id"] = rng.choice(shared_devices, size=len(shared_device_idx), replace=True)
mark_scenario(shared_device_idx, "shared_device_ring")

# 2. Beneficiary concentrator
remaining_idx = transacoes_fraud.index[transacoes_fraud["is_fraud"] == 0]
beneficiary_idx = rng.choice(remaining_idx, size=1_400, replace=False)
concentrator_beneficiaries = rng.choice(beneficiarios["beneficiario_id"], size=20, replace=False)
transacoes_fraud.loc[beneficiary_idx, "beneficiario_id"] = rng.choice(concentrator_beneficiaries, size=len(beneficiary_idx), replace=True)
transacoes_fraud.loc[beneficiary_idx, "valor"] = np.round(rng.lognormal(mean=7.1, sigma=0.7, size=len(beneficiary_idx)), 2)
mark_scenario(beneficiary_idx, "beneficiary_concentrator")

# 3. New account high value
remaining_idx = transacoes_fraud.index[transacoes_fraud["is_fraud"] == 0]
new_account_idx = rng.choice(remaining_idx, size=1_000, replace=False)

if len(recent_accounts) > 0:
    transacoes_fraud.loc[new_account_idx, "conta_origem_id"] = rng.choice(recent_accounts, size=len(new_account_idx), replace=True)

transacoes_fraud.loc[new_account_idx, "valor"] = np.round(rng.uniform(8_000, 45_000, size=len(new_account_idx)), 2)
transacoes_fraud.loc[new_account_idx, "tipo_transacao"] = rng.choice(["pix", "ted"], size=len(new_account_idx), p=[0.80, 0.20])
mark_scenario(new_account_idx, "new_account_high_value")

# 4. Burst transactions
remaining_idx = transacoes_fraud.index[transacoes_fraud["is_fraud"] == 0]
burst_idx = rng.choice(remaining_idx, size=1_200, replace=False)
burst_accounts = rng.choice(active_accounts, size=40, replace=False)
burst_start = pd.Timestamp("2025-11-20 21:00:00")

transacoes_fraud.loc[burst_idx, "conta_origem_id"] = rng.choice(burst_accounts, size=len(burst_idx), replace=True)
transacoes_fraud.loc[burst_idx, "data_hora"] = [
    burst_start + timedelta(minutes=int(x))
    for x in rng.integers(0, 180, size=len(burst_idx))
]
transacoes_fraud.loc[burst_idx, "valor"] = np.round(rng.uniform(300, 3_500, size=len(burst_idx)), 2)
transacoes_fraud.loc[burst_idx, "tipo_transacao"] = "pix"
mark_scenario(burst_idx, "burst_transactions")

# 5. Bridge account
remaining_idx = transacoes_fraud.index[transacoes_fraud["is_fraud"] == 0]
bridge_idx = rng.choice(remaining_idx, size=800, replace=False)
bridge_accounts = rng.choice(active_accounts, size=12, replace=False)

transacoes_fraud.loc[bridge_idx, "conta_origem_id"] = rng.choice(bridge_accounts, size=len(bridge_idx), replace=True)
transacoes_fraud.loc[bridge_idx, "valor"] = np.round(rng.uniform(1_000, 12_000, size=len(bridge_idx)), 2)
transacoes_fraud.loc[bridge_idx, "tipo_transacao"] = rng.choice(["pix", "transferencia_interna"], size=len(bridge_idx), p=[0.70, 0.30])
mark_scenario(bridge_idx, "bridge_account")

# 6. Coordinated network
remaining_idx = transacoes_fraud.index[transacoes_fraud["is_fraud"] == 0]
network_idx = rng.choice(remaining_idx, size=1_000, replace=False)
network_accounts = rng.choice(active_accounts, size=60, replace=False)
network_devices = rng.choice(high_risk_devices, size=10, replace=False)
network_ips = rng.choice(high_risk_ips, size=10, replace=False)
network_beneficiaries = rng.choice(beneficiarios["beneficiario_id"], size=15, replace=False)

transacoes_fraud.loc[network_idx, "conta_origem_id"] = rng.choice(network_accounts, size=len(network_idx), replace=True)
transacoes_fraud.loc[network_idx, "device_id"] = rng.choice(network_devices, size=len(network_idx), replace=True)
transacoes_fraud.loc[network_idx, "ip_id"] = rng.choice(network_ips, size=len(network_idx), replace=True)
transacoes_fraud.loc[network_idx, "beneficiario_id"] = rng.choice(network_beneficiaries, size=len(network_idx), replace=True)
transacoes_fraud.loc[network_idx, "valor"] = np.round(rng.uniform(700, 18_000, size=len(network_idx)), 2)
mark_scenario(network_idx, "coordinated_network")

transacoes_fraud["data_hora"] = pd.to_datetime(transacoes_fraud["data_hora"])
transacoes_fraud = transacoes_fraud.sort_values("data_hora").reset_index(drop=True)

transacoes_fraud.head()

,transacao_id,conta_origem_id,beneficiario_id,valor,data_hora,tipo_transacao,canal,device_id,ip_id,status_transacao,is_fraud,fraud_scenario,cartao_id
0,TX_00007707,CTA_001877,BEN_003322,203.30,2025-01-01 03:05:51,pix,app,DEV_001817,IP_000089,aprovada,0,normal,None
1,TX_00072064,CTA_003365,BEN_000265,275.99,2025-01-01 03:13:25,pix,app,DEV_002968,IP_001202,aprovada,0,normal,None
2,TX_00075764,CTA_000156,BEN_002832,11.59,2025-01-01 04:00:23,pix,app,DEV_003600,IP_001519,aprovada,0,normal,None
3,TX_00023203,CTA_005491,BEN_001801,296.97,2025-01-01 04:08:36,pix,app,DEV_001308,IP_001428,aprovada,0,normal,None
4,TX_00001302,CTA_000997,BEN_002808,94.98,2025-01-01 04:15:04,boleto,api,DEV_003878,IP_002967,aprovada,0,normal,None


## 8. Criação da Tabela de Labels

A tabela `labels_fraude` separa os rótulos analíticos das transações.

Essa separação é útil porque permite manter a tabela de transações como evento operacional e a tabela de labels como camada analítica de validação e treinamento.

In [8]:
labels_fraude = transacoes_fraud[
    ["transacao_id", "is_fraud", "fraud_scenario"]
].copy()

labels_fraude["label_descricao"] = np.where(
    labels_fraude["is_fraud"] == 1,
    "Transação sintética marcada como suspeita/fraudulenta para fins analíticos.",
    "Transação sintética com comportamento considerado normal.",
)

labels_fraude.head()

,transacao_id,is_fraud,fraud_scenario,label_descricao
0,TX_00007707,0,normal,Transação sintética com comportamento considerado normal.
1,TX_00072064,0,normal,Transação sintética com comportamento considerado normal.
2,TX_00075764,0,normal,Transação sintética com comportamento considerado normal.
3,TX_00023203,0,normal,Transação sintética com comportamento considerado normal.
4,TX_00001302,0,normal,Transação sintética com comportamento considerado normal.


## 9. Validações Iniciais da Base Sintética

Antes de exportar os dados, serão executadas validações simples para verificar:

- quantidade de registros por dataset;
- taxa geral de fraude sintética;
- distribuição dos cenários;
- valores mínimos, médios e máximos;
- integridade básica entre transações e entidades.

In [9]:
datasets_summary = pd.DataFrame(
    [
        {"dataset": "clientes", "linhas": len(clientes), "colunas": clientes.shape[1]},
        {"dataset": "contas", "linhas": len(contas), "colunas": contas.shape[1]},
        {"dataset": "dispositivos", "linhas": len(dispositivos), "colunas": dispositivos.shape[1]},
        {"dataset": "ips", "linhas": len(ips), "colunas": ips.shape[1]},
        {"dataset": "beneficiarios", "linhas": len(beneficiarios), "colunas": beneficiarios.shape[1]},
        {"dataset": "cartoes", "linhas": len(cartoes), "colunas": cartoes.shape[1]},
        {"dataset": "transacoes", "linhas": len(transacoes_fraud), "colunas": transacoes_fraud.shape[1]},
        {"dataset": "labels_fraude", "linhas": len(labels_fraude), "colunas": labels_fraude.shape[1]},
    ]
)

fraud_rate = transacoes_fraud["is_fraud"].mean()

print(f"Taxa sintética de fraude: {fraud_rate:.2%}")

datasets_summary

Taxa sintética de fraude: 8.75%


,dataset,linhas,colunas
0,clientes,5000,6
1,contas,6000,6
2,dispositivos,4500,4
3,ips,3000,4
4,beneficiarios,3500,4
5,cartoes,4000,5
6,transacoes,80000,13
7,labels_fraude,80000,4


In [10]:
scenario_distribution = (
    transacoes_fraud["fraud_scenario"]
    .value_counts()
    .rename_axis("fraud_scenario")
    .reset_index(name="qtd_transacoes")
)

scenario_distribution["percentual"] = (
    scenario_distribution["qtd_transacoes"] / scenario_distribution["qtd_transacoes"].sum()
).round(4)

scenario_distribution

,fraud_scenario,qtd_transacoes,percentual
0,normal,73000,0.9125
1,shared_device_ring,1600,0.0200
2,beneficiary_concentrator,1400,0.0175
3,burst_transactions,1200,0.0150
4,new_account_high_value,1000,0.0125
5,coordinated_network,1000,0.0125
6,bridge_account,800,0.0100


In [11]:
transaction_value_summary = (
    transacoes_fraud
    .groupby("fraud_scenario")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        valor_min=("valor", "min"),
        valor_medio=("valor", "mean"),
        valor_mediano=("valor", "median"),
        valor_max=("valor", "max"),
        taxa_fraude=("is_fraud", "mean"),
    )
    .reset_index()
)

transaction_value_summary[["valor_min", "valor_medio", "valor_mediano", "valor_max"]] = (
    transaction_value_summary[["valor_min", "valor_medio", "valor_mediano", "valor_max"]].round(2)
)

transaction_value_summary

,fraud_scenario,qtd_transacoes,valor_min,valor_medio,valor_mediano,valor_max,taxa_fraude
0,beneficiary_concentrator,1400,95.27,1586.40,1230.71,8670.90,1.0
1,bridge_account,800,1006.89,6556.05,6651.77,11984.34,1.0
2,burst_transactions,1200,301.72,1875.91,1815.92,3499.92,1.0
3,coordinated_network,1000,714.04,9359.03,9387.64,17980.38,1.0
4,new_account_high_value,1000,8085.11,26752.50,26593.05,44999.68,1.0
5,normal,73000,5.00,472.21,272.78,25000.00,0.0
6,shared_device_ring,1600,8.44,441.32,264.50,9947.93,1.0


## 10. Validação de Integridade Referencial

Nesta etapa verificamos se as chaves usadas nas transações existem nas tabelas de entidades.

Essa validação é importante porque os próximos notebooks dependerão desses relacionamentos para criar regras, features e conexões no grafo.

In [12]:
integrity_checks = {
    "conta_origem_id_em_contas": transacoes_fraud["conta_origem_id"].isin(contas["conta_id"]).mean(),
    "beneficiario_id_em_beneficiarios": transacoes_fraud["beneficiario_id"].isin(beneficiarios["beneficiario_id"]).mean(),
    "device_id_em_dispositivos": transacoes_fraud["device_id"].isin(dispositivos["device_id"]).mean(),
    "ip_id_em_ips": transacoes_fraud["ip_id"].isin(ips["ip_id"]).mean(),
}

integrity_df = pd.DataFrame(
    [{"check": key, "percentual_valido": value} for key, value in integrity_checks.items()]
)

integrity_df

,check,percentual_valido
0,conta_origem_id_em_contas,1.0
1,beneficiario_id_em_beneficiarios,1.0
2,device_id_em_dispositivos,1.0
3,ip_id_em_ips,1.0


## 11. Exportação dos Dados Sintéticos

Os datasets serão exportados em formato Parquet para a pasta:

`data/synthetic/`

Essa pasta está ignorada pelo Git para evitar versionamento de dados gerados. O código do notebook permite recriar os dados sempre que necessário.

Também será criado um manifesto em Markdown na pasta `docs/`, documentando os arquivos gerados.

In [13]:
exports = {
    "clientes": clientes,
    "contas": contas,
    "dispositivos": dispositivos,
    "ips": ips,
    "beneficiarios": beneficiarios,
    "cartoes": cartoes,
    "transacoes": transacoes_fraud,
    "labels_fraude": labels_fraude,
}

for name, df in exports.items():
    output_path = SYNTHETIC_DIR / f"{name}.parquet"
    df.to_parquet(output_path, index=False)

print("Arquivos Parquet exportados:")
for file in sorted(SYNTHETIC_DIR.glob("*.parquet")):
    print(f"- {file.name}")

Arquivos Parquet exportados:
- beneficiarios.parquet
- cartoes.parquet
- clientes.parquet
- contas.parquet
- dispositivos.parquet
- ips.parquet
- labels_fraude.parquet
- transacoes.parquet


In [14]:
manifest_lines = [
    "# Manifesto dos Dados Sintéticos",
    "",
    "Este documento registra os datasets sintéticos gerados pelo Notebook 01.",
    "",
    "## Arquivos Gerados",
    "",
]

for name, df in exports.items():
    manifest_lines.append(f"### `{name}.parquet`")
    manifest_lines.append("")
    manifest_lines.append(f"- Linhas: {len(df):,}".replace(",", "."))
    manifest_lines.append(f"- Colunas: {df.shape[1]}")
    manifest_lines.append(f"- Caminho: `data/synthetic/{name}.parquet`")
    manifest_lines.append("")

manifest_lines.extend(
    [
        "## Distribuição dos Cenários de Fraude",
        "",
        scenario_distribution.to_markdown(index=False),
        "",
        "## Observação",
        "",
        "Os dados são sintéticos e foram criados exclusivamente para fins educacionais, analíticos e de portfólio.",
        "Nenhum dado real de cliente, instituição financeira ou transação foi utilizado.",
        "",
    ]
)

manifest_path = DOCS_DIR / "synthetic_data_manifest.md"
manifest_path.write_text("\n".join(manifest_lines), encoding="utf-8")

print(f"Manifesto criado em: {manifest_path}")

Manifesto criado em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs\synthetic_data_manifest.md


## 12. Conclusão Executiva do Notebook 01

Este notebook criou a primeira versão dos dados sintéticos do projeto **Fraud Graph Analytics**.

Foram geradas as principais entidades do domínio:

- clientes;
- contas;
- dispositivos;
- IPs;
- beneficiários;
- cartões;
- transações;
- labels de fraude.

Também foram injetados cenários suspeitos controlados, incluindo:

- dispositivo compartilhado;
- beneficiário concentrador;
- conta nova com alto valor;
- transações em rajada;
- conta ponte;
- rede coordenada.

Esses dados serão usados nos próximos notebooks para:

- análise exploratória transacional;
- criação de regras antifraude explicáveis;
- construção do Knowledge Graph;
- aplicação de métricas de centralidade e comunidades;
- criação de score de risco antifraude.

O próximo passo será o **Notebook 02 — EDA Transactional Fraud**, onde os dados sintéticos serão explorados para identificar padrões, distribuições, outliers e sinais iniciais de risco.